<a href="https://colab.research.google.com/github/sugandhi15/emailSpamDetector/blob/main/mailSpamDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [73]:
import kagglehub
path = kagglehub.dataset_download("venky73/spam-mails-dataset")

Using Colab cache for faster access to the 'spam-mails-dataset' dataset.


In [74]:
print(path)

import os

print(os.listdir(path))

/kaggle/input/spam-mails-dataset
['spam_ham_dataset.csv']


In [75]:
file_path = "/root/.cache/kagglehub/datasets/venky73/spam-mails-dataset/versions/1/spam_ham_dataset.csv"

In [76]:
import pandas as pd

df = pd.read_csv(file_path)

print(df.head())

   Unnamed: 0 label                                               text  \
0         605   ham  Subject: enron methanol ; meter # : 988291\r\n...   
1        2349   ham  Subject: hpl nom for january 9 , 2001\r\n( see...   
2        3624   ham  Subject: neon retreat\r\nho ho ho , we ' re ar...   
3        4685  spam  Subject: photoshop , windows , office . cheap ...   
4        2030   ham  Subject: re : indian springs\r\nthis deal is t...   

   label_num  
0          0  
1          0  
2          0  
3          1  
4          0  


In [77]:
df.drop(columns=["Unnamed: 0"], inplace=True)

In [78]:
print(df.head())

  label                                               text  label_num
0   ham  Subject: enron methanol ; meter # : 988291\r\n...          0
1   ham  Subject: hpl nom for january 9 , 2001\r\n( see...          0
2   ham  Subject: neon retreat\r\nho ho ho , we ' re ar...          0
3  spam  Subject: photoshop , windows , office . cheap ...          1
4   ham  Subject: re : indian springs\r\nthis deal is t...          0


In [79]:
df.drop(columns=["label"], inplace=True)

In [80]:
df.head()

,text,label_num
0,Subject: enron methanol ; meter # : 988291\r\n...,0
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,"Subject: photoshop , windows , office . cheap ...",1
4,Subject: re : indian springs\r\nthis deal is t...,0


In [81]:
import re
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [82]:
def preprocess_text(text):
  text.lower()

  text = re.sub(r'^[a-zA-Z]','',text)

  token = word_tokenize(text)

  token = [word for word in token if word not in stopwords.words('english')]

  stemmer = PorterStemmer()

  token = [stemmer.stem(word) for word in token]

  return ' '.join(token)

In [83]:
df['text'] = df['text'].apply(preprocess_text)

In [84]:
print(df.head())

                                                text  label_num
0  ubject : enron methanol ; meter # : 988291 fol...          0
1  ubject : hpl nom januari 9 , 2001 ( see attach...          0
2  ubject : neon retreat ho ho ho , ' around wond...          0
3  ubject : photoshop , window , offic . cheap . ...          1
4  ubject : : indian spring deal book teco pvr re...          0


In [85]:
# Convert Text to Numbers
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)

x = tfidf.fit_transform(df['text']).toarray()

y = df['label_num']

In [86]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    x,y,test_size=0.2,random_state=42
)

In [87]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train,y_train)

MultinomialNB()

In [88]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)

print(accuracy_score(y_test,pred))

0.9526570048309179


In [89]:
msgs = [
    "FREE entry into our prize draw. Claim now!",
    "URGENT! You have won a lottery worth 50000",
    "Call now to receive your free gift"
]

msgs = [preprocess_text(x) for x in msgs]
msgs = tfidf.transform(msgs)

print(model.predict(msgs))
print(model.predict_proba(msgs))

[1 1 1]
[[0.26075874 0.73924126]
 [0.21157763 0.78842237]
 [0.44759229 0.55240771]]


In [90]:
test_msgs = [
    "Congratulations! You have won a free iPhone. Claim now!",
    "URGENT! Your account has been selected for a cash prize.",
    "Win ₹50000 instantly by clicking this link.",
    "Hey, are we still meeting tomorrow?",
    "Can you send me the notes from today's class?",
    "Mom said dinner is ready.",
    "Free entry in a weekly competition. Text WIN to 12345.",
    "You have been awarded a lottery worth ₹1,00,000.",
    "Good morning, have a nice day!",
    "Please call me when you reach home."
]

# Preprocess
processed_msgs = [preprocess_text(msg) for msg in test_msgs]

# Convert to TF-IDF
X_test_custom = tfidf.transform(processed_msgs)

# Predict
predictions = model.predict(X_test_custom)

# Print results
for msg, pred in zip(test_msgs, predictions):
    label = "Spam" if pred == 1 else "Ham"
    print(f"{label}: {msg}")

Spam: Congratulations! You have won a free iPhone. Claim now!
Spam: URGENT! Your account has been selected for a cash prize.
Spam: Win ₹50000 instantly by clicking this link.
Ham: Hey, are we still meeting tomorrow?
Ham: Can you send me the notes from today's class?
Ham: Mom said dinner is ready.
Spam: Free entry in a weekly competition. Text WIN to 12345.
Spam: You have been awarded a lottery worth ₹1,00,000.
Ham: Good morning, have a nice day!
Ham: Please call me when you reach home.


In [91]:
import pickle

pickle.dump(model, open("spam_model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf.pkl", "wb"))

In [100]:
!ls

app.py	  node_modules	package-lock.json  spam_model.pkl
logs.txt  package.json	sample_data	   tfidf.pkl


In [125]:
!pip install streamlit -q
!pip install streamlit nltk scikit-learn
!npm install -g localtunnel

In [96]:
%%writefile app.py

import streamlit as st
import pickle
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Ensure required NLTK data is present (safe to call repeatedly, quiet=True suppresses noise)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# Cache model/vectorizer loading so it only happens once per session
@st.cache_resource
def load_artifacts():
    model = pickle.load(open("spam_model.pkl", "rb"))
    tfidf = pickle.load(open("tfidf.pkl", "rb"))
    return model, tfidf

model, tfidf = load_artifacts()

STOP_WORDS = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Text preprocessing
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in STOP_WORDS]
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)


# Streamlit UI
st.set_page_config(
    page_title="SMS Spam Detector",
    page_icon="📩"
)

st.title("📩 SMS Spam Detector")

st.write(
    "Enter a message below and the model will predict whether it is Spam or Ham."
)

message = st.text_area(
    "Enter SMS Message"
)

if st.button("Check Message"):

    if message.strip() == "":
        st.warning("Please enter a message.")
    else:

        cleaned_message = preprocess_text(message)

        vector = tfidf.transform([cleaned_message])

        prediction = model.predict(vector)[0]

        probability = model.predict_proba(vector)[0]

        if prediction == 1:
            st.error("🚨 Spam Message")
            st.write(
                f"Spam Probability: {probability[1] * 100:.2f}%"
            )
        else:
            st.success("✅ Ham Message")
            st.write(
                f"Ham Probability: {probability[0] * 100:.2f}%"
            )

        with st.expander("Processed Text"):
            st.write(cleaned_message)

Overwriting app.py


In [126]:
!ls

app.py	  node_modules	package-lock.json  spam_model.pkl
logs.txt  package.json	sample_data	   tfidf.pkl


In [128]:
!pkill -f streamlit
!pkill -f lt

In [ ]:
!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501

⠙⠹your url is: https://seven-colts-peel.loca.lt
